## Simple Java

Java is a wonderful language. For people like me who graduated from college in early 2000s, it was the primary language that had a very high chance to land us into the professional space. 

### First Java Program

As is the customary in any language, we should start first by writing a program that prints "Hello World" to the console. So how is that program going to look like ? 

In [15]:
// Save the file as chap01/MyHelloWorld.java 
class HelloWorld {

    public static void main(String[] args){

        System.out.println("My Hello World");
    }

}

Had we been in the terminal we could have just executed the below command to execute this program and see the output. 

*java HelloWorld.java*

But we are in a Java Notebook. Yes, just like a Jupyter Notebook, Oracle has facilitated us to create Java Notebooks similar to the way one would use a Jupyter notebook for Python. 

But to run an external file like the one above *MyHelloWorld.java* we need to do the below
- Create a process in Java that will run the java command
- The process to use the full file path as an argument
- Start the process, which in effect would run our program
- The program prints to the console, so the console output needs to be captured back
- The captured back stream is printed to the console just like any other *System.out.println*

So, the program that does all of that is below. 

In [16]:
import java.io.*;

String filePath = "/home/arnabp/projects/simple_java/src/main/java/com/arnabp/chap01/HelloWorld.java";

ProcessBuilder pb = new ProcessBuilder("java", filePath);
pb.redirectErrorStream(true);
Process process = pb.start();

try (BufferedReader reader = new BufferedReader(new InputStreamReader(process.getInputStream()))) {
    String line;
    while ((line = reader.readLine()) != null) {
        System.out.println(line);
    }
}
process.waitFor();


My Hello World


This file when executed produces the output *My Hello World*. 

But this could also be achieved using a more easier approach, now that we have understood that this content is created using a Java notebook. The approach is as below
- Create a class with the same name MyHelloWorld
- Create a method with the name sayHello that will print the same string. Keep in mind that we are **NOT** using the main method here, as this is not required. 
- Create an instance of the class named helloWorld

In [17]:
public class MyHelloWorld {
    public void sayHello (){
        System.out.println("My Hello World");
    }
}

System.out.println("class MyHelloWorld is created ..."); 

class MyHelloWorld is created ...


Now if I want to call the above program to check the output, I would do the below 
- create a variable named helloWorld to hold the instance of the MyHelloWorld class
- call the *sayHello* method on that instance

So, here you are ... 

In [18]:
var helloWorld = new MyHelloWorld();
helloWorld.sayHello();

My Hello World


#### Static Methods

The static methods in Java, just like the main method does **NOT** require an object to be instantiated from the class to be called and used. They can be called directly by using the `ClassName.staticMethodName` syntax. The example code is as below. 

In [19]:
public class HelloWorldStatic {
    
    public static void sayHello(){

        System.out.println("Hello World from Static");

    }

}

HelloWorldStatic.sayHello();


Hello World from Static


### Data Types in Java

We will now discuss about the methods `parseInt` and `valueOf` of the Integer class. Let us first of all see how they both work. 

- `Integer.parseInt`

This method is used to parse a string that looks like an integer. So, for example as below. Please remeber that here `parseInt` converts the String into a *primitive* integer.  


In [20]:
var myIntPrimitive = Integer.parseInt("100");
System.out.println("myIntPrimitive = " + myIntPrimitive);    

myIntPrimitive = 100


`Integer.valueOf`

This method can also pass the string to an integer. So, for example as below. But here we need to remember that this method returns an *Integer Object* and not a *primitive* integer. 

In [21]:
var myIntObject = Integer.valueOf("100");

System.out.println("myIntObject = " + myIntObject);

myIntObject = 100


But how do we test our understanding that `Integer.parseInt` returns a primitive integer and `Integer.valueOf` returns an Integer object. Let us use the concept of *Method Overloading* to confirm our understanding. The code block is here with the associated outputs. 

In [22]:
public static boolean isPrimitive(int x) {
    return true;
}

public static boolean isPrimitive(Integer x){
    return false;
}

if (isPrimitive(myIntObject)){
    System.out.println("myIntObject is Primitive");
} else if (!isPrimitive(myIntObject)){
    System.out.println("myIntObject is Object");
}

if (isPrimitive(myIntPrimitive)){
    System.out.println("myIntPrimitive is Primitive");
} else if (!isPrimitive(myIntPrimitive)){
    System.out.println("myIntPrimitive is Object");
}



myIntObject is Object
myIntPrimitive is Primitive


So, there we are. The seemingly same methods `Integer.parseInt` and `Integer.valueOf` have different return types. The key points are 
- If you only need a primitive `int`, `parseInt` is straightforward and efficient. 
- If you need an `Integer` instance (for *Autoboxing* or APIs requiring Integer objects), use `valueOf`. 

### Threads in Java 

Threads in Java are quite interesting and is the core of *Concurrency* in Java. The Modern Java has progressed from defining classes that implements Runnable and then creating Thread objects and starting them. Modern Java now has `ExecutorService` and `Executors` to submit jobs and `Future` to keep track of them and ensure that these jobs are closed and the computed objects retrieved for subsequent processing. 
So let us solve a typical problem where we need to sum first N natural numbers by dividing this set to to parts - the first part involving the numbers starting from 1 and to the mean of the range and the second part beginning immediately next to the mean number and finishing at the end. So if we have to sum first 20 natural numbers, the first part will be 1 to 10 and the second part will be 11 to 20. These two parts need to be summed in separate individual threads and then when these both threads complete the results from them are to summed to arrive at the final result of the sum. How can this be done in Java using the modern concpets of Java? The code snippet below attempts to do this. And also we will get to touch upon the beauty of Java Streams API, which we will discuss later on in this notebook with some more examples.  

In [23]:
import java.util.stream.IntStream;
import java.util.stream.Collectors;
import java.util.List;
import java.util.concurrent.ExecutorService;
import java.util.concurrent.Executors;
import java.util.concurrent.Callable; 
import java.util.concurrent.Future;
// import java.util.concurrent.TimeUnit;


class DoSumInThread implements Callable<Integer> {
    
    private ArrayList<Integer> listToSum; 
    private int threadNum;
    private int result = 0;

    DoSumInThread(ArrayList<Integer> listToSum, int threadNum){
        this.listToSum = listToSum;
        this.threadNum = threadNum;
    }

    
    @Override
    public Integer call() {


        int tempSum = 0;

        System.out.println("Starting Thread " + threadNum + " ... ");

        try {
            
                for (int i = 0; i < listToSum.size(); i++){
                
                System.out.println("Iterating Thead " + threadNum + " with iteration number " + (i + 1));
                tempSum = tempSum + listToSum.get(i); 
                Thread.sleep(1000);              
             }
        } catch (InterruptedException e){
            Thread.currentThread().interrupt();
            System.out.println("Thread " + threadNum + " is interrupted");
        }    

        result = tempSum;
        System.out.println("Thread " + threadNum + " sum = " + result);    
        return result;
    }

} 

// Let the numbers to sum be 20 

int rangeEnd = 20;

int firstThreadEndsOn = rangeEnd / 2;
int secondThreadStartsOn = firstPart + 1;


var executor = Executors.newFixedThreadPool(2);

// declare the Thread 1 
var listSumOne = new ArrayList<Integer>(IntStream.rangeClosed(1, firstThreadEndsOn).boxed().collect(Collectors.toList()));

Future<Integer> doSumInThreadOne = executor.submit(new DoSumInThread(listSumOne, 1));

// declare the Thread 2 
var listSumTwo = new ArrayList<Integer>(IntStream.rangeClosed(secondThreadStartsOn, rangeEnd).boxed().collect(Collectors.toList()));

Future<Integer> doSumInThreadTwo = executor.submit(new DoSumInThread(listSumTwo, 2));


try {
    
    int resultOne = doSumInThreadOne.get();
    int resultTwo = doSumInThreadTwo.get();

    System.out.println("Both threads finished successfully");
    System.out.println("Final Result = " + (resultOne + resultTwo));

} catch (Exception e) {
    e.printStackTrace();
} finally {
    executor.shutdown();
}


cannot find symbol
  symbol:   variable firstPart
int secondThreadStartsOn = firstPart + 1;
                           ^-------^
cannot find symbol
  symbol:   variable secondThreadStartsOn
var listSumTwo = new ArrayList<Integer>(IntStream.rangeClosed(secondThreadStartsOn, rangeEnd).boxed().collect(Collectors.toList()));
                                                              ^------------------^
cannot find symbol
  symbol:   variable listSumTwo
Future<Integer> doSumInThreadTwo = executor.submit(new DoSumInThread(listSumTwo, 2));
                                                                     ^--------^
cannot find symbol
  symbol:   variable doSumInThreadTwo
    int resultTwo = doSumInThreadTwo.get();
                    ^--------------^
Starting Thread 1 ... 
Iterating Thead 1 with iteration number 1


### Data Visualization Project (Modern Java - Together Java)
We will now look at an interesting project in the book 

[Modern Java - Together Java - Data Visualization](https://javabook.mccue.dev/projects/data_visualization)

This project captured my imagination. I never heard about a proposition to create a data visualization using Core Java. The project is an inttersting reading. I urge you to read the project very carefully, and just a moment to ponder and imagine how we can create a data visulaization without using any native packages of Java or any data visualization software. 

The PPM P3 image format and the details about this can be found in the Wikipedia Page here. I urge you to read this.  

[Wikipedia - Netpbm - Details of PPM P3 Image Format ](https://en.wikipedia.org/wiki/Netpbm)

The program implementation of this in Java for an example string is as given below. 


In [24]:
import java.util.Map;
import java.util.function.Function;
import java.util.stream.Collectors;
import java.nio.file.Files;
import java.nio.file.Path;
import java.io.IOException;

public class PPMObject {

    private String digitString = "";
    private Map<Character, Long> digitCountsMap;
    private record colorCombo(int red, int green, int blue) {};
    private ArrayList<colorCombo> myColors = new ArrayList<>(Arrays.asList(
            new colorCombo(112, 175, 144), 
            new colorCombo(159, 183, 234), 
            new colorCombo(45, 24, 41), 
            new colorCombo(255, 140, 0), 
            new colorCombo(11, 17, 7), 
            new colorCombo(86, 114, 41), 
            new colorCombo(134, 133, 140), 
            new colorCombo(200, 50, 100), 
            new colorCombo(155, 154, 153), 
            new colorCombo(255, 215, 04)
    ));
    private colorCombo blackPixelColor = new colorCombo(255, 255, 255);
    private String textForZero = "";
    private String textForOne = "";
    private String textForTwo = "";
    private String textForThree = "";
    private String textForFour = "";
    private String textForFive = "";
    private String textForSix = "";
    private String textForSeven = "";
    private String textForEight = "";
    private String textForNine = "";
    
    private Long maxValue = 0L;
    
    


    PPMObject (String digitString){
        this.digitString = digitString;
        this.digitCountsMap = this.digitString.chars()
                                    .mapToObj(c -> (char) c)
                                    .collect(Collectors.groupingBy(
                                        Function.identity(), 
                                        Collectors.counting()
                                    ));
        
        this.setMaxValue();                            
                                    
    }

    
    public void setMaxValue() {
        for (Map.Entry<Character, Long> entry: digitCountsMap.entrySet()){
            
            if (entry.getValue() > maxValue)
                this.maxValue = entry.getValue();
        }
    }

    public Map<Character, Long> getDigitCountsMap(){
        return this.digitCountsMap;
    }

    public String createColorTexts(int digitKey, Long digitValue) {

        String returnString = "";

        colorCombo cmbTemplateDigitValue = myColors.get(digitKey);

        String textToAdd = cmbTemplateDigitValue.red + " " + cmbTemplateDigitValue.green + " " + cmbTemplateDigitValue.blue;
        String blackPixel = blackPixelColor.red + " " + blackPixelColor.green + " " + blackPixelColor.blue;

        for (int iters=0; iters < maxValue; iters++){
        
            if (iters < digitValue){
                for (int row=0; row < 20; row++){
                    for (int col=0; col < 15; col++){
                        returnString = returnString + textToAdd + " ";
                    }

                    returnString = returnString + "\n";

                }

            } else {
                
                for (int row=0; row < 20; row++){
                    for (int col=0; col < 15; col++){
                        returnString = returnString + blackPixel + " ";
                    }

                    returnString = returnString + "\n";

                }   
            }
        }    

        return returnString;
    }

    public void createFileForDigits() throws IOException{

        
        for (Map.Entry<Character, Long> entry: this.digitCountsMap.entrySet()){
            
            int key = Character.getNumericValue(entry.getKey());
            Long value = entry.getValue();
            
            if (key == 0){
                
                this.textForZero = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForZero.txt"), this.textForZero);
                
            } else if (key == 1){
                
                this.textForOne = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForOne.txt"), this.textForOne);
            
            } else if (key == 2){
            
                this.textForTwo = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForTwo.txt"), this.textForTwo);
            
            } else if (key == 3){
            
                this.textForThree = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForThree.txt"), this.textForThree);
            
            } else if (key == 4){
            
                this.textForFour = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForFour.txt"), this.textForFour);
            
            } else if (key == 5){
            
                this.textForFive = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForFive.txt"), this.textForFive);
            
            } else if (key == 6){
            
                this.textForSix = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForSix.txt"), this.textForSix);
            
            } else if (key == 7){
            
                this.textForSeven = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForSeven.txt"), this.textForSeven);
            
            } else if (key == 8){
            
                this.textForEight = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForEight.txt"), this.textForEight);
            
            } else if (key == 9){
            
                this.textForNine = this.createColorTexts(key, value);
                Files.writeString(Path.of("fileForNine.txt"), this.textForNine);
            
            } 


        }

    }
};


PPMObject myPPMObject = new PPMObject("124124523950159219359858327587324573258723458342756734");

System.out.println(myPPMObject.getDigitCountsMap());

myPPMObject.createFileForDigits();


{0=1, 1=4, 2=9, 3=8, 4=6, 5=10, 6=1, 7=6, 8=5, 9=4}
Iterating Thead 1 with iteration number 2


The above program create a separate file for every digit in as much as files with name fileForZero.txt, fileForOne.txt ... fileForNine.txt. 
The next logical step is to concatenate these files side by side into one single file with the P3 format headers. The below `FileConcatenator` class does this. This file concatenation side by side can also be accomplished in a more sleak way by using the below command at a Linux Terminal 

`paste -d "" fileForZero.txt fileForOne.txt fileForTwo.txt fileForThree.txt fileForFour.txt fileForFive.txt fileForSix.txt fileForSeven.txt fileForEight.txt fileForNine.txt > final_output.ppm`


In [25]:
// File Concatenation Program 

import java.io.*;

public class FileConcatenator {
    public static void concatenateFiles() {
        String rootPath = "/home/arnabp/projects/simple_java/src/main/java/com/arnabp/";
        String file0 = rootPath + "fileForZero.txt";
        String file1 = rootPath + "fileForOne.txt";
        String file2 = rootPath + "fileForTwo.txt";
        String file3 = rootPath + "fileForThree.txt";
        String file4 = rootPath + "fileForFour.txt";
        String file5 = rootPath + "fileForFive.txt";
        String file6 = rootPath + "fileForSix.txt";
        String file7 = rootPath + "fileForSeven.txt";
        String file8 = rootPath + "fileForEight.txt";
        String file9 = rootPath + "fileForNine.txt";
        
        String outputFile = rootPath + "final_output.ppm";
               

        try (BufferedReader br0 = new BufferedReader(new FileReader(file0));
             BufferedReader br1 = new BufferedReader(new FileReader(file1));
             BufferedReader br2 = new BufferedReader(new FileReader(file2));
             BufferedReader br3 = new BufferedReader(new FileReader(file3));
             BufferedReader br4 = new BufferedReader(new FileReader(file4));
             BufferedReader br5 = new BufferedReader(new FileReader(file5));
             BufferedReader br6 = new BufferedReader(new FileReader(file6));
             BufferedReader br7 = new BufferedReader(new FileReader(file7));
             BufferedReader br8 = new BufferedReader(new FileReader(file8));
             BufferedReader br9 = new BufferedReader(new FileReader(file9));
             PrintWriter pw = new PrintWriter(new FileWriter(outputFile));
             ) {

            String line0, line1, line2, line3, line4, line5, line6, line7, line8, line9;
            pw.println("P3");
            pw.println("# A simple 150x200 image");
            pw.println("150 200");
            pw.println("255");
            
            // Read both files line by line simultaneously
            while ((line0 = br0.readLine()) != null && 
                    (line1 = br1.readLine()) != null && 
                    (line2 = br2.readLine()) != null && 
                    (line3 = br3.readLine()) != null && 
                    (line4 = br4.readLine()) != null && 
                    (line5 = br5.readLine()) != null && 
                    (line6 = br6.readLine()) != null && 
                    (line7 = br7.readLine()) != null && 
                    (line8 = br8.readLine()) != null && 
                    (line9 = br9.readLine()) != null
                    ) {
                // Combine lines with your choice of separator (e.g., "")
                pw.println(line0 + " " + line1 + " " + line2 + " " + line3 + " " + line4 + " " + line5 + " " + line6 + " " + line7 + " " + line8 + " " + line9 );
            }
        } catch (IOException e) {
            e.printStackTrace();
        }
    }
}

// Create the final output file 
FileConcatenator.concatenateFiles();

The data visualization created from the above implementation looks as below

![Data Viz](final_output_ppm.png)